# 🎬 Netflix Content Dashboard (Interactive)

An interactive **Plotly Dash** dashboard built from `netflix_titles.csv` (Kaggle — shivamb),
styled after the dark red/black Netflix Power BI reference dashboard.

**Layout:** header + KPI cards, filter panel, top-countries chart, movie/TV ratio donut,
year-over-year trend, top genres, rating distribution, and a searchable titles table.

**Filters (all live-update the whole page):** Type (Movie/TV Show), Country, Rating, Release Year range.

**How to use:**
1. Make sure `netflix_titles.csv` is in `/mnt/user-data/uploads/` (or edit `DATA_DIR` in the app cell below).
2. Run all cells top to bottom.
3. The last cell starts the Dash app **inline** in the notebook output — interact with the filters there.


In [ ]:
import subprocess, sys
# Uncomment if dash / dash-bootstrap-components aren't installed in your environment:
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'dash', 'dash-bootstrap-components', '-q'])


## Build the interactive dashboard app

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, dash_table
import dash_bootstrap_components as dbc

# ---------------------------------------------------------
# BRAND PALETTE
# ---------------------------------------------------------
RED    = '#E50914'
DARKBG = '#141414'
PANEL  = '#221F1F'
WHITE  = '#F5F5F1'
GREY   = '#8a8a8a'
PALETTE = [RED, '#B20710', '#831010', WHITE, GREY, '#F5C518']

# ---------------------------------------------------------
# LOAD & CLEAN DATA
# ---------------------------------------------------------
DATA_DIR = '/mnt/user-data/uploads/'
raw = pd.read_csv(DATA_DIR + 'netflix_titles.csv')

df = raw.copy()
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['country_primary'] = df['country'].fillna('Unknown').apply(lambda x: str(x).split(',')[0].strip())
df['rating'] = df['rating'].fillna('Not Rated')

genre_df = df.assign(genre=df['listed_in'].str.split(', ')).explode('genre')

TYPES = sorted(df['type'].dropna().unique())
COUNTRIES = df['country_primary'].value_counts().head(30).index.tolist()
RATINGS = sorted(df['rating'].dropna().unique())
YEAR_MIN, YEAR_MAX = int(df['release_year'].min()), int(df['release_year'].max())

# ---------------------------------------------------------
# APP
# ---------------------------------------------------------
app = Dash(__name__, external_stylesheets=[dbc.themes.CYBORG], title='Netflix Content Dashboard')
server = app.server


def kpi_card(title, value_id):
    return dbc.Card(
        dbc.CardBody([
            html.H3(id=value_id, style={'fontWeight': '800', 'color': WHITE, 'margin': '0'}),
            html.Div(title, style={'fontSize': '11px', 'color': GREY, 'fontWeight': '600',
                                    'textTransform': 'uppercase', 'letterSpacing': '.6px', 'marginTop': '4px'})
        ]),
        style={'backgroundColor': PANEL, 'borderRadius': '10px', 'border': f'1px solid {RED}',
               'textAlign': 'center'}
    )


header = html.Div([
    html.Div('NETFLIX', style={'color': RED, 'fontWeight': '900', 'fontSize': '26px', 'letterSpacing': '1px'}),
    html.Div('Movies & TV Shows Data Analysis', style={'color': WHITE, 'fontSize': '13px'}),
], style={'backgroundColor': PANEL, 'borderRadius': '10px', 'padding': '18px', 'textAlign': 'center',
          'height': '100%'})

filters = html.Div([
    html.Div('FILTERS', style={'color': RED, 'fontWeight': '700', 'fontSize': '12px', 'marginBottom': '10px'}),
    html.Label('Type', style={'color': WHITE, 'fontSize': '11px'}),
    dcc.Dropdown(id='f-type', options=[{'label': t, 'value': t} for t in TYPES],
                 multi=True, placeholder='All', style={'marginBottom': '10px', 'color': '#111'}),
    html.Label('Country', style={'color': WHITE, 'fontSize': '11px'}),
    dcc.Dropdown(id='f-country', options=[{'label': c, 'value': c} for c in COUNTRIES],
                 multi=True, placeholder='All', style={'marginBottom': '10px', 'color': '#111'}),
    html.Label('Rating', style={'color': WHITE, 'fontSize': '11px'}),
    dcc.Dropdown(id='f-rating', options=[{'label': r, 'value': r} for r in RATINGS],
                 multi=True, placeholder='All', style={'marginBottom': '10px', 'color': '#111'}),
    html.Label('Release Year', style={'color': WHITE, 'fontSize': '11px'}),
    dcc.RangeSlider(id='f-year', min=YEAR_MIN, max=YEAR_MAX, value=[YEAR_MIN, YEAR_MAX],
                     marks={YEAR_MIN: str(YEAR_MIN), YEAR_MAX: str(YEAR_MAX)},
                     tooltip={'placement': 'bottom', 'always_visible': False}),
], style={'backgroundColor': PANEL, 'borderRadius': '10px', 'padding': '16px', 'height': '100%'})

app.layout = html.Div([
    dbc.Row([
        dbc.Col(header, width=3),
        dbc.Col(kpi_card('Total Titles', 'kpi-total'), width=2),
        dbc.Col(kpi_card('Movies', 'kpi-movies'), width=2),
        dbc.Col(kpi_card('TV Shows', 'kpi-tv'), width=2),
        dbc.Col(kpi_card('Countries', 'kpi-countries'), width=3),
    ], className='g-2', style={'marginBottom': '10px'}),

    dbc.Row([
        dbc.Col(filters, width=3),
        dbc.Col(dcc.Graph(id='fig-countries', config={'displayModeBar': False}), width=6),
        dbc.Col(dcc.Graph(id='fig-ratio', config={'displayModeBar': False}), width=3),
    ], className='g-2', style={'marginBottom': '10px'}),

    dbc.Row([
        dbc.Col(dcc.Graph(id='fig-trend', config={'displayModeBar': False}), width=6),
        dbc.Col(dcc.Graph(id='fig-genre', config={'displayModeBar': False}), width=6),
    ], className='g-2', style={'marginBottom': '10px'}),

    dbc.Row([
        dbc.Col(dcc.Graph(id='fig-rating', config={'displayModeBar': False}), width=6),
        dbc.Col(html.Div([
            html.Div('TITLES', style={'color': RED, 'fontWeight': '700', 'fontSize': '12px', 'marginBottom': '8px'}),
            dash_table.DataTable(
                id='tbl-titles', page_size=8,
                style_header={'backgroundColor': RED, 'color': WHITE, 'fontWeight': '700', 'fontSize': '11px'},
                style_cell={'backgroundColor': PANEL, 'color': WHITE, 'fontSize': '11px',
                            'textAlign': 'left', 'padding': '6px', 'maxWidth': '180px',
                            'overflow': 'hidden', 'textOverflow': 'ellipsis'},
                style_data_conditional=[{'if': {'row_index': 'odd'}, 'backgroundColor': '#2b2727'}],
            )
        ], style={'backgroundColor': PANEL, 'borderRadius': '10px', 'padding': '14px', 'height': '100%'}), width=6),
    ], className='g-2'),

], style={'backgroundColor': DARKBG, 'minHeight': '100vh', 'padding': '14px',
          'fontFamily': 'Segoe UI, Arial, sans-serif'})


def style_ax(fig, title):
    fig.update_layout(
        title=dict(text=title, font=dict(size=14, color=WHITE, family='Segoe UI', weight='bold')),
        paper_bgcolor=PANEL, plot_bgcolor=PANEL,
        font=dict(color=WHITE, family='Segoe UI'),
        margin=dict(l=40, r=20, t=44, b=30), height=300,
    )
    return fig


def apply_filters(types, countries, ratings, years):
    d = df.copy()
    if types:
        d = d[d['type'].isin(types)]
    if countries:
        d = d[d['country_primary'].isin(countries)]
    if ratings:
        d = d[d['rating'].isin(ratings)]
    d = d[(d['release_year'] >= years[0]) & (d['release_year'] <= years[1])]
    return d


@app.callback(
    Output('kpi-total', 'children'), Output('kpi-movies', 'children'),
    Output('kpi-tv', 'children'), Output('kpi-countries', 'children'),
    Output('fig-countries', 'figure'), Output('fig-ratio', 'figure'),
    Output('fig-trend', 'figure'), Output('fig-genre', 'figure'),
    Output('fig-rating', 'figure'),
    Output('tbl-titles', 'data'), Output('tbl-titles', 'columns'),
    Input('f-type', 'value'), Input('f-country', 'value'),
    Input('f-rating', 'value'), Input('f-year', 'value'),
)
def update_dashboard(types, countries, ratings, years):
    d = apply_filters(types, countries, ratings, years)
    ids = set(d['show_id'])
    g = genre_df[genre_df['show_id'].isin(ids)]

    total = len(d)
    movies = (d['type'] == 'Movie').sum()
    tv = (d['type'] == 'TV Show').sum()
    n_countries = d['country_primary'].nunique()

    # Top countries
    top_c = d['country_primary'].value_counts().head(10).sort_values()
    fig_countries = go.Figure(go.Bar(x=top_c.values, y=top_c.index, orientation='h',
                                      marker=dict(color=RED)))
    style_ax(fig_countries, 'Top Content-Producing Countries')

    # Movie/TV ratio
    ratio = d['type'].value_counts()
    fig_ratio = go.Figure(go.Pie(labels=ratio.index, values=ratio.values, hole=0.55,
                                  marker=dict(colors=[RED, WHITE])))
    style_ax(fig_ratio, 'Movies vs TV Shows')

    # Trend
    trend = d.dropna(subset=['year_added']).groupby(['year_added', 'type']).size().unstack(fill_value=0)
    fig_trend = go.Figure()
    colors_map = {'Movie': RED, 'TV Show': WHITE}
    for col in trend.columns:
        fig_trend.add_trace(go.Scatter(x=trend.index, y=trend[col], mode='lines+markers',
                                        name=col, line=dict(color=colors_map.get(col, GREY), width=3)))
    style_ax(fig_trend, 'Titles Added Per Year')
    fig_trend.update_layout(legend=dict(bgcolor=PANEL, font=dict(color=WHITE)))

    # Genre
    top_g = g['genre'].value_counts().head(10).sort_values()
    fig_genre = go.Figure(go.Bar(x=top_g.values, y=top_g.index, orientation='h',
                                  marker=dict(color=WHITE)))
    style_ax(fig_genre, 'Top 10 Genres')

    # Rating distribution
    rate_counts = d['rating'].value_counts().head(10)
    fig_rating = go.Figure(go.Bar(x=rate_counts.index, y=rate_counts.values,
                                   marker=dict(color=PALETTE * 3)))
    style_ax(fig_rating, 'Content Rating Distribution')

    tbl = d[['rating', 'type', 'title', 'country_primary']].rename(
        columns={'country_primary': 'country'}).head(300)
    columns = [{'name': c, 'id': c} for c in tbl.columns]
    data = tbl.to_dict('records')

    return (f"{total:,}", f"{movies:,}", f"{tv:,}", f"{n_countries}",
            fig_countries, fig_ratio, fig_trend, fig_genre, fig_rating, data, columns)


## Run the dashboard
Running this cell launches the dashboard **inline, right below the cell**. Use the filter panel to slice the data — every chart, KPI and table updates live.

In [ ]:
# Runs inline inside Jupyter (JupyterLab / Notebook / VS Code).
# If you're in a plain Jupyter Notebook and 'inline' mode doesn't render, change jupyter_mode to 'external'
# and open the printed http://127.0.0.1:8051 link in your browser instead.
app.run(debug=False, port=8051, jupyter_mode='inline', jupyter_height=1000)
